# Tiago Oliveira Lima - 564988

In [74]:
import numpy as np
from typing import Literal
from abc import ABC, abstractmethod

# Código

## BaseModel

In [75]:
class BaseModel(ABC):
    def __init__(self):
        self.X_train = None
        self.y_train = None

    @abstractmethod
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    @abstractmethod
    def predict(self, X):
        pass

    def evaluate(self, X_test, y_test):
        y_pred = self.predict(X_test)
        
        return {
            "accuracy": self.accuracy(y_test, y_pred),
            "precision": self.precision(y_test, y_pred),
            "recall": self.recall(y_test, y_pred),
            "f1_score": self.f1_score(y_test, y_pred)
        }

    @staticmethod
    def accuracy(y_true, y_pred):
        return np.mean(y_true == y_pred)

    def precision(self, y_true, y_pred):
        cm = self.confusion_matrix(y_true, y_pred)
        precisions = []
        for i in range(len(cm)):
            tp = cm[i, i]
            fp = np.sum(cm[:, i]) - tp
            if (tp + fp) > 0:
                precisions.append(tp / (tp + fp))
            else:
                precisions.append(0)
        return np.mean(precisions)

    def recall(self, y_true, y_pred):
        cm = self.confusion_matrix(y_true, y_pred)
        recalls = []
        for i in range(len(cm)):
            tp = cm[i, i]
            fn = np.sum(cm[i, :]) - tp
            if (tp + fn) > 0:
                recalls.append(tp / (tp + fn))
            else:
                recalls.append(0)
        return np.mean(recalls)

    def f1_score(self, y_true, y_pred):
        p = self.precision(y_true, y_pred)
        r = self.recall(y_true, y_pred)
        if (p + r) > 0:
            return 2 * (p * r) / (p + r)
        return 0

    @staticmethod
    def confusion_matrix(y_true, y_pred):
        classes = np.unique(np.concatenate([y_true, y_pred]))
        n_classes = len(classes)
        label_to_idx = {val: idx for idx, val in enumerate(classes)}
        
        matrix = np.zeros((n_classes, n_classes), dtype=int)
        for true, pred in zip(y_true, y_pred):
            matrix[label_to_idx[true], label_to_idx[pred]] += 1
        return matrix
    
    def cross_validate(self, X: np.ndarray, y: np.ndarray, folds: int = 5):
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]
        X_folds, y_folds = np.array_split(X, folds), np.array_split(y, folds)
        
        fold_metrics = []
        for i in range(folds):
            X_val, y_val = X_folds[i], y_folds[i]
            X_train_f = np.concatenate([X_folds[j] for j in range(folds) if j != i])
            y_train_f = np.concatenate([y_folds[j] for j in range(folds) if j != i])
            self.fit(X_train_f, y_train_f)
            fold_metrics.append(self.evaluate(X_val, y_val))
            
        return {m: np.mean([f[m] for f in fold_metrics]) for m in fold_metrics[0]}

## KNN

In [76]:
class KNN(BaseModel):
    def __init__(self, k: int = 3, metric: Literal["euclidean", "mahalanobis"] = "euclidean"):
        super().__init__() 
        self.k = k
        self.metric = metric
        self.inv_cov = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        super().fit(X, y)
        
        if self.metric == "mahalanobis":
            cov = np.cov(X, rowvar=False) 
            self.inv_cov = np.linalg.pinv(cov) 
        return self 

    def predict(self, X: np.ndarray):
        if self.X_train is None:
            raise ValueError("Modelo não treinado.")
        return np.array([self._predict(x) for x in X])

    def _predict(self, x):
        if self.metric == "euclidean":
            distances = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
        elif self.metric == "mahalanobis":
            delta = self.X_train - x
            distances = np.sqrt(np.sum(np.dot(delta, self.inv_cov) * delta, axis=1))
        
        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = self.y_train[k_indices]
        
        vals, counts = np.unique(k_nearest_labels, return_counts=True)
        return vals[counts.argmax()]

## Decision Trees

In [77]:
from sklearn.tree import DecisionTreeClassifier

class DecisionTree(BaseModel):
    def __init__(self, criterion: Literal["gini", "entropy"] = "gini", max_depth=None):
        super().__init__()
        self.criterion = criterion
        self.max_depth = max_depth
        self.model = DecisionTreeClassifier(criterion=self.criterion, max_depth=self.max_depth)

    def fit(self, X: np.ndarray, y: np.ndarray):
        super().fit(X, y)
        self.model.fit(X, y)
        return self

    def predict(self, X: np.ndarray):
        if self.X_train is None:
            raise ValueError("Modelo não treinado.")
        return self.model.predict(X)

# Treinando modelos


## Treinando knn

In [78]:
arquivo = 'kc2.csv'

X = np.genfromtxt(arquivo, delimiter=',', skip_header=1, usecols=range(21), dtype=float)

y_str = np.genfromtxt(arquivo, delimiter=',', skip_header=1, usecols=21, dtype=str)

classes, y = np.unique(y_str, return_inverse=True)

print(f"Dataset carregado: {X.shape[0]} amostras e {X.shape[1]} atributos.")
print(f"Classes identificadas: {classes} mapeadas para {np.unique(y)}")

Dataset carregado: 213 amostras e 21 atributos.
Classes identificadas: ['0.000000000000000000e+00' '1.000000000000000000e+00'] mapeadas para [0 1]


### Distancia euclidiana

In [79]:
modelo_euclidean = KNN(k=10, metric="euclidean")

resultados = modelo_euclidean.cross_validate(X, y, folds=5)

print("--- Relatório de Performance euclidean ---")
for metrica, valor in resultados.items():
    print(f"{metrica.capitalize()}: {valor:.4f}")

--- Relatório de Performance euclidean ---
Accuracy: 0.7742
Precision: 0.7694
Recall: 0.7801
F1_score: 0.7746


### Distancia de mahalanobis

In [80]:
modelo_mahalanobis = KNN(k=10, metric="mahalanobis")

resultados = modelo_mahalanobis.cross_validate(X, y, folds=5)

print("--- Relatório de Performance Mahalanobis ---")
for metrica, valor in resultados.items():
    print(f"{metrica.capitalize()}: {valor:.4f}")

--- Relatório de Performance Mahalanobis ---
Accuracy: 0.7093
Precision: 0.7370
Recall: 0.7143
F1_score: 0.7253


## Treinando arvore de decisão

In [81]:
# Gini
arvore_gini = DecisionTree(criterion="gini", max_depth=5)
res_gini = arvore_gini.cross_validate(X, y, folds=5)

# Entropia
arvore_entropy = DecisionTree(criterion="entropy", max_depth=5)
res_entropy = arvore_entropy.cross_validate(X, y, folds=5)

print("--- RESULTADOS (MÉDIA DOS FOLDS) ---")
print(f"Gini DT    - F1: {res_gini['f1_score']:.4f} | Acc: {res_gini['accuracy']:.4f}")
print(f"Entropy DT - F1: {res_entropy['f1_score']:.4f} | Acc: {res_entropy['accuracy']:.4f}")

--- RESULTADOS (MÉDIA DOS FOLDS) ---
Gini DT    - F1: 0.7440 | Acc: 0.7467
Entropy DT - F1: 0.7643 | Acc: 0.7651


## Comparando knn com arvore de decisão

In [82]:

metricas = ("accuracy", "precision", "recall", "f1_score")

def uma_linha(nome: str, res: dict) -> str:
    return (
        f"{nome:<34}"
        + "".join(f"{res[m]:>12.4f}" for m in metricas)
    )

np.random.seed(42)
res_knn_euc = KNN(k=10, metric="euclidean").cross_validate(X, y, folds=10)
np.random.seed(42)
res_knn_mah = KNN(k=10, metric="mahalanobis").cross_validate(X, y, folds=10)
np.random.seed(42)
res_dt_gini = DecisionTree(criterion="gini", max_depth=5).cross_validate(X, y, folds=10)
np.random.seed(42)
res_dt_ent = DecisionTree(criterion="entropy", max_depth=5).cross_validate(X, y, folds=10)

linhas = [
    ("KNN (k=10, Euclidiana)", res_knn_euc),
    ("KNN (k=10, Mahalanobis)", res_knn_mah),
    ("Árvore — Gini (prof. max 10)", res_dt_gini),
    ("Árvore — Entropia (prof. max 10)", res_dt_ent),
]

cab = (
    f"{'Modelo':<34}"
    + f"{'Acurácia':>12}{'Precisão':>12}{'Revocação':>12}{'F1-score':>12}"
)
print(cab)
print("-" * len(cab))
for nome, res in linhas:
    print(uma_linha(nome, res))


Modelo                                Acurácia    Precisão   Revocação    F1-score
----------------------------------------------------------------------------------
KNN (k=10, Euclidiana)                  0.7602      0.7649      0.7592      0.7619
KNN (k=10, Mahalanobis)                 0.7275      0.7619      0.7405      0.7509
Árvore — Gini (prof. max 10)            0.7703      0.7803      0.7761      0.7781
Árvore — Entropia (prof. max 10)        0.7600      0.7654      0.7590      0.7621
